A notebook for prompting of SOTA LLMs.

## Data Loading

In [9]:

import json
import re

import os
from pathlib import Path

from dataset_processing import process_llm_jsonl_results, shorten_name

import time

with open("not_to_upload.txt", "r") as f:
    API_KEY = f.read().split("\n")
    
with open("system_prompt.md", "r") as f:
    system_prompt = f.read()
    
with open("definitions.txt", "r") as f:
    definitions_str = f.read()

definitions_dict = [{"label": definition.split(" --- ")[0], "definition":definition.split(" --- ")[1]} for definition in definitions_str.split("\n")]

def create_string(def_dict):
    definitions_string = ""
    for ent in def_dict:
        definitions_string = definitions_string + ent["label"] + " - " + ent["definition"] + "\n"
        
    return definitions_string

system_prompt_withdefinitions = re.sub(r"\[TAXONOMY - DEFINITIONS - RULES\]", create_string(definitions_dict), system_prompt)

In [10]:
DATA_DIR = "RESULTS/annots_v2"

In [11]:
files = os.listdir(DATA_DIR)

sentences = {}

for file in files:
    file_dir = Path.joinpath(Path(DATA_DIR),Path(file))
    print(file_dir)

    with open(file_dir, "r") as f:
        data_json = json.load(f)
        
    for i in range(len(data_json)):
        paper_id = data_json[i]["data"]["paper_id"]
        sentence_id = data_json[i]["data"]["sentence_id"]
        
        compound = f"{paper_id}-{sentence_id}"
        # print(compound)
        sentences[f"{compound}"]= data_json[i]["data"]["sentence"]

print(len(sentences))



RESULTS/annots_v2/mmp_50_MatExp_MeaDev_PhyPhe_Qua.json
RESULTS/annots_v2/mmp_50_EneSou_MetPhe_NatDis_NatPhe.json
RESULTS/annots_v2/mmp_50_Ass_Pol.json
RESULTS/annots_v2/mmp_50_PhyArt_Org_Per_Sys.json
RESULTS/annots_v2/mmp_50_Loc_GeoFea_BodOfWat_TimPer_Sat.json
RESULTS/annots_v2/mmp_50_BodPar_Che_Dis_Org_Eco.json
RESULTS/annots_v2/mmp_50_Met_FieOfStu_IntArt.json
192


### Helper Functions

In [12]:
MODEL_NAME = "Unnamed"
def query_llm_gemini(client, input_sentence, system_prompt, model_name=MODEL_NAME):
    """
    Sends the sentence to Gemini and returns a dictionary containing 
    the input and the raw response text.
    """
    try:
        response = client.models.generate_content(
            model=model_name,
            contents=input_sentence,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                response_mime_type="application/json", 
                temperature=0.1
            )
        )
        
        # Return a structured package
        return {
            "input_text": input_sentence,
            "raw_response": response.text,
            "success": True
        }
        
    except Exception as e:
        print(f"Error processing sentence: {input_sentence[:30]}... | Error: {e}")
        return {
            "input_text": input_sentence,
            "raw_response": None,
            "success": False,
            "error": str(e)
        }

def query_llm_openai(client_openai, input_sentence, system_prompt, model_name=MODEL_NAME):
    """
    Sends the sentence to OpenAI and returns a structured package 
    compatible with the existing parse_and_align_spans function.
    """
    try:
        response = client_openai.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": input_sentence}
            ],
            # NOTE: If your system prompt asks for a LIST [...], remove response_format.
            # If it asks for an OBJECT {"entities": [...]}, keep it.
            # OpenAI 'json_object' mode strictly requires the output to be a dictionary {}.
            response_format={"type": "json_object"}, 
            temperature=0.1
        )
        
        raw_content = response.choices[0].message.content
        
        return {
            "input_text": input_sentence,
            "raw_response": raw_content,
            "success": True
        }
        
    except Exception as e:
        print(f"Error processing sentence: {input_sentence[:30]}... | Error: {e}")
        return {
            "input_text": input_sentence,
            "raw_response": None,
            "success": False,
            "error": str(e)
        }

def parse_and_align_spans(result_package):
    """
    Parses the JSON response and calculates exact start/end indices.
    It maintains a cursor position to handle identical entities appearing twice.
    """
    if not result_package["success"] or not result_package["raw_response"]:
        return []

    input_text = result_package["input_text"]
    raw_json = result_package["raw_response"]
    
    extracted_entities = []

    try:
        # 1. Clean and Parse JSON
        # (Sometimes models add ```json markdown, though mime_type usually prevents it)
        clean_json = raw_json.replace("```json", "").replace("```", "").strip()
        data = json.loads(clean_json)

        # 2. Iterative Span Finding
        current_cursor = 0
        
        for item in data:
            entity_text = item.get('entity_text', '').strip()
            category = item.get('category', 'Unknown')
            reasoning = item.get('reasoning', '')

            if not entity_text:
                continue

            # Escape the entity text for Regex (handles parens, dots, etc.)
            pattern = re.escape(entity_text)
            
            # Search ONLY in the part of the string after the current cursor
            match = re.search(pattern, input_text[current_cursor:])
            
            if match:
                # Calculate absolute indices based on the cursor offset
                relative_start = match.start()
                relative_end = match.end()
                
                abs_start = current_cursor + relative_start
                abs_end = current_cursor + relative_end
                
                extracted_entities.append({
                    "entity_text": entity_text,
                    "category": category,
                    "reasoning": reasoning,
                    "span": (abs_start, abs_end)
                })
                
                # Update cursor so next search starts AFTER this entity
                current_cursor = abs_end
            else:
                print(f"Warning: Could not strictly align '{entity_text}' after index {current_cursor}.")
                
                # Optional: specific fallback search from index 0
                fallback_match = re.search(pattern, input_text)
                if fallback_match:
                    extracted_entities.append({
                        "entity_text": entity_text,
                        "category": category,
                        "span": (fallback_match.start(), fallback_match.end()),
                        "note": "Out of order match"
                    })

    except json.JSONDecodeError:
        print("JSON Parsing failed.")
    
    return extracted_entities



## Gemini

In [14]:
# !pip install google-genai
from google import genai
from google.genai import types

In [19]:
client = genai.Client(api_key=API_KEY[0])
SYSTEM_INSTRUCTION = system_prompt_withdefinitions
MODEL_NAME = "gemini-2.5-pro"

RPM = 25
RPD = 1000000  
OUTPUT_FILE = f"ner_results_{shorten_name(MODEL_NAME)}.jsonl"

sleeptime = (60 / RPM) + 1 
requests_count = 0


In [ ]:
sentences_to_process = list(sentences.keys()) # specific list of keys to control order if needed

print(f"Starting job. Est. time: {len(sentences_to_process[:RPD]) * sleeptime / 60:.1f} minutes.")

# Open file in APPEND mode so we don't overwrite if we restart the script
with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:
    
    for i, sentence_id in enumerate(sentences_to_process):
        
        # 1. Check Hard Limit
        if requests_count >= RPD:
            print("Daily/Batch limit reached. Stopping.")
            break

        sentence = sentences[sentence_id]
        print(f"[{i+1}/{min(len(sentences), RPD)}] Processing ID {sentence_id}...")

        # 2. Step A: Query
        # (Assuming query_llm_gemini is defined as in the previous turn)
        raw_result = query_llm_gemini(client, sentence, SYSTEM_INSTRUCTION, MODEL_NAME)
        
        # 3. Step B: Parse
        # Only parse if the query was successful
        if raw_result['success']:
            final_entities = parse_and_align_spans(raw_result)
        else:
            final_entities = [] # Or handle error specifically

        # 4. Step C: Prepare Entry
        entry = {
            "compound_id": sentence_id,
            "input_text": sentence,
            # Don't save the full raw_result string if you want to save space, 
            # but it is good for debugging.
            "raw_llm_response": raw_result.get('raw_response'), 
            "processed_entities": final_entities,
            "timestamp": time.time()
        }

        # 5. Write to File IMMEDIATELY (JSONL format)
        # This saves your data even if the script crashes next second.
        f_out.write(json.dumps(entry) + "\n")
        f_out.flush() # Force write to disk

        # 6. Rate Limiting
        requests_count += 1
        time.sleep(sleeptime)

print(f"\nDone. Results saved to {OUTPUT_FILE}")

Starting job. Est. time: 10.9 minutes.
[1/192] Processing ID 62269-66...
[2/192] Processing ID 6641-129...
[3/192] Processing ID 40290-2...
[4/192] Processing ID 40290-19...
[5/192] Processing ID 99773-22...
[6/192] Processing ID 75726-11...
[7/192] Processing ID 84006-44...
[8/192] Processing ID 90439-25...
[9/192] Processing ID 62269-117...
[10/192] Processing ID 28901-8...
[11/192] Processing ID 62269-73...
[12/192] Processing ID 10662-66...
[13/192] Processing ID 35800-4...
[14/192] Processing ID 10662-18...
[15/192] Processing ID 39322-79...
[16/192] Processing ID 56596-53...
[17/192] Processing ID 84006-38...
[18/192] Processing ID 46034-54...
[19/192] Processing ID 16783-1601...
[20/192] Processing ID 12778-11...
[21/192] Processing ID 6641-1...
[22/192] Processing ID 10662-54...
[23/192] Processing ID 2416-0...
[24/192] Processing ID 99773-39...


## ChatGPT

In [ ]:
# !pip install openai
from openai import OpenAI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 7.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [openai]2m2/3 [openai]


In [8]:
SYSTEM_INSTRUCTION

'### ROLE\nYou are an expert Named Entity Recognition (NER) system. Your task is to extract entities from the user\'s input text and classify them according to the provided taxonomy.\n\n### TAXONOMY & RULES\nAsset - is an object or service of value to humans that can get destroyed or diminished by climate disasters/hazards. Key categories are health, buildings, infrastructure, and crops or livestock.\nBody of Water - is a distinct volume or mass of water, whether naturally occurring (like an ocean or river) or contained within a man-made system (like a reservoir or effluent flow). This includes specific named bodies, general types, and scientifically defined water masses.\nBody Part - is a structural component of a living organism (plant, animal, or human), which is not the whole organism but a distinct anatomical or morphological part.\nChemical - is a substance with a distinct molecular composition, including elements, compounds, ions, biological molecules, and mixtures. It also enco

In [7]:
# 1. Setup
client = OpenAI(api_key=API_KEY[1]) # Using index 1 as requested

SYSTEM_INSTRUCTION = system_prompt_withdefinitions
MODEL_NAME = "gpt-4o" # or "gpt-4-turbo", "gpt-3.5-turbo"

RPM = 500
RPD = 1
OUTPUT_FILE = f"ner_results_{shorten_name(MODEL_NAME)}.jsonl"

sleeptime = (60 / RPM) + 1 
requests_count = 0

sentences_to_process = list(sentences.keys()) 

print(f"Starting job. Est. time: {len(sentences_to_process[:RPD]) * sleeptime / 60:.1f} minutes.")

# --- MAIN EXECUTION LOOP ---

# Open file in APPEND mode
with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:
    
    for i, sentence_id in enumerate(sentences_to_process):
        
        # 1. Check Hard Limit
        if requests_count >= RPD:
            print("Daily/Batch limit reached. Stopping.")
            break

        sentence = sentences[sentence_id]
        print(f"[{i+1}/{min(len(sentences), RPD)}] Processing ID {sentence_id}...")

        # 2. Step A: Query
        raw_result = query_llm_openai(client, sentence, SYSTEM_INSTRUCTION, MODEL_NAME)
        
        # 3. Step B: Parse
        # Using the same parser function from the previous setup
        if raw_result['success']:
            # Note: If OpenAI returns a wrapper dict (e.g. {"items": [...]}) due to json_object mode,
            # ensure parse_and_align_spans handles dictionary inputs, or use raw_result['raw_response'] logic.
            final_entities = parse_and_align_spans(raw_result)
        else:
            final_entities = []

        # 4. Step C: Prepare Entry
        entry = {
            "compound_id": sentence_id,
            "input_text": sentence,
            "raw_llm_response": raw_result.get('raw_response'), 
            "processed_entities": final_entities,
            "timestamp": time.time()
        }

        # 5. Write to File IMMEDIATELY
        f_out.write(json.dumps(entry) + "\n")
        f_out.flush()

        # 6. Rate Limiting
        requests_count += 1
        time.sleep(sleeptime)

print(f"\nDone. Results saved to {OUTPUT_FILE}")

Starting job. Est. time: 0.0 minutes.
[1/1] Processing ID 62269-66...


AttributeError: 'str' object has no attribute 'get'

In [ ]:
# 1. Setup
client = OpenAI(api_key=API_KEY[1])

system_instruction = system_prompt_withdefinitions

# 2. User Input
user_input = "The experimental results showed that the pH levels dropped by 50% in the Danube River."

# 3. ChatGPT API Call with JSON enforcement
response = client.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_input}
    ],
    response_format={"type": "json_object"},  # Force valid JSON
    temperature=0.1
)

# 4. Extract JSON text
response_json = response.choices[0].message.content

# 5. Process Result
try:
    data = json.loads(response_json)

    # Example logic to find spans (naive string match)
    for item in data:
        entity = item["entity_text"]
        start_index = user_input.find(entity)
        if start_index != -1:
            end_index = start_index + len(entity)
            print(f"Found '{entity}' ({item['category']}) at [{start_index}:{end_index}]")
            print(f"Reason: {item['reasoning']}\n")
        else:
            print(f"Warning: Could not locate '{entity}' exactly in source.")

except json.JSONDecodeError:
    print("Model failed to generate valid JSON.")
    print("Raw output:\n", response_json)


## DeepSeek

In [31]:
from openai import OpenAI

In [35]:

# 1. Setup
client = OpenAI(
    api_key=API_KEY[2],
    base_url="https://api.deepseek.com"
)

system_instruction = system_prompt_withdefinitions

# 2. User Input
user_input = "The experimental results showed that the pH levels dropped by 50% in the Danube River."

# 3. Call API with JSON enforcement
response = client.chat.completions.create(
    model="deepseek-chat",  # or "deepseek-coder" for coding tasks
    messages=[
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_input}
    ],
    response_format={"type": "json_object"},  # Forces valid JSON output
    temperature=0.1,  # Low temp is better for extraction tasks
    stream=False
)

# 4. Process Result
try:
    # Extract the JSON response
    response_text = response.choices[0].message.content
    data = json.loads(response_text)
    
    # Example logic to find spans (naive string match)
    for item in data:
        entity = item['entity_text']
        start_index = user_input.find(entity)
        if start_index != -1:
            end_index = start_index + len(entity)
            print(f"Found '{entity}' ({item['category']}) at [{start_index}:{end_index}]")
            print(f"Reason: {item['reasoning']}\n")
        else:
            print(f"Warning: Could not locate '{entity}' exactly in source.")
            
except json.JSONDecodeError:
    print("Model failed to generate valid JSON.")
    print(response_text)
except Exception as e:
    print(f"Error: {e}")

APIStatusError: Error code: 402 - {'error': {'message': 'Insufficient Balance', 'type': 'unknown_error', 'param': None, 'code': 'invalid_request_error'}}